<a href="https://colab.research.google.com/github/maragadhavelt/DAA/blob/main/Sales_Performance_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy plotly scikit-learn statsmodels mlxtend xgboost

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from statsmodels.tsa.arima.model import ARIMA
from mlxtend.frequent_patterns import apriori, association_rules
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

np.random.seed(10)

n = 6000

orders = pd.DataFrame({
    "Order ID": np.arange(50000,50000+n),
    "Customer ID": np.random.randint(100,1500,n),
    "Product": np.random.choice(["Laptop","Mobile","Tablet","Camera","TV","Watch","Headphones"],n),
    "Region": np.random.choice(["North","South","East","West"],n),
    "Channel": np.random.choice(["Online","Offline"],n),
    "Order Date": pd.date_range(start="2020-01-01", periods=n, freq="D"),
    "Sales": np.random.randint(5000,200000,n),
    "Cost": np.random.randint(2000,120000,n)
})

customers = pd.DataFrame({
    "Customer ID": np.arange(100,1500),
    "Age": np.random.randint(18,65,1400),
    "Gender": np.random.choice(["M","F"],1400),
    "Income": np.random.randint(20000,150000,1400)
})

products = pd.DataFrame({
    "Product":["Laptop","Mobile","Tablet","Camera","TV","Watch","Headphones"],
    "Category":["Electronics","Electronics","Electronics","Electronics","Electronics","Accessories","Accessories"]
})

df = orders.merge(customers, on="Customer ID").merge(products, on="Product")

df = df[df["Sales"] > df["Cost"]]

df["Profit"] = df["Sales"] - df["Cost"]
df["Margin"] = df["Profit"]/df["Sales"]

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Week"] = df["Order Date"].dt.isocalendar().week
df["Day"] = df["Order Date"].dt.day

df["Lag1"] = df["Sales"].shift(1)
df["Lag7"] = df["Sales"].shift(7)
df["Rolling7"] = df["Sales"].rolling(7).mean()

df = df.fillna(method="bfill")

customer = df.groupby("Customer ID").agg({
    "Sales":"sum",
    "Profit":"sum",
    "Order ID":"count"
}).reset_index()

scaler = StandardScaler()
scaled = scaler.fit_transform(customer[["Sales","Profit","Order ID"]])

kmeans = KMeans(n_clusters=5)
customer["Segment"] = kmeans.fit_predict(scaled)

X_reg = df[["Cost","Lag1","Lag7","Rolling7"]]
y_reg = df["Sales"]

rf_reg = RandomForestRegressor()
rf_reg.fit(X_reg,y_reg)
pred_reg = rf_reg.predict(X_reg)

xgb_reg = xgb.XGBRegressor()
xgb_reg.fit(X_reg,y_reg)
xgb_pred = xgb_reg.predict(X_reg)

rmse_rf = np.sqrt(mean_squared_error(y_reg,pred_reg))
rmse_xgb = np.sqrt(mean_squared_error(y_reg,xgb_pred))

df["High Profit"] = (df["Profit"] > df["Profit"].median()).astype(int)

X_clf = df[["Sales","Cost","Margin"]]
y_clf = df["High Profit"]

clf = RandomForestClassifier()
clf.fit(X_clf,y_clf)
pred_clf = clf.predict(X_clf)

acc = accuracy_score(y_clf,pred_clf)

log = LogisticRegression()
log.fit(X_clf,y_clf)

monthly = df.groupby("Month")["Sales"].sum()
arima = ARIMA(monthly, order=(2,1,2)).fit()
forecast = arima.forecast(6)

basket = df.groupby(["Order ID","Product"])["Sales"].sum().unstack().fillna(0)
basket = basket.applymap(lambda x: 1 if x>0 else 0)

freq = apriori(basket, min_support=0.05, use_colnames=True)
rules = association_rules(freq, metric="lift", min_threshold=1)

recommend = rules.sort_values("lift", ascending=False).head(10)

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled)

cohort = df.copy()
cohort["Cohort"] = cohort.groupby("Customer ID")["Order Date"].transform("min").dt.to_period("M")
cohort["Order Period"] = cohort["Order Date"].dt.to_period("M")
cohort["Index"] = (cohort["Order Period"] - cohort["Cohort"]).apply(lambda x: x.n)

retention = cohort.groupby(["Cohort","Index"])["Customer ID"].nunique().reset_index()

pivot = pd.pivot_table(df, values="Sales", index="Region", columns="Product", aggfunc=np.sum)

corr = df[["Sales","Cost","Profit","Margin"]].corr()

fig1 = px.line(monthly)
fig2 = px.scatter(customer, x="Sales", y="Profit", color="Segment")
fig3 = px.bar(recommend, x="antecedents", y="lift")
fig4 = px.scatter(x=pca_data[:,0], y=pca_data[:,1], color=customer["Segment"])

fig5 = go.Figure()
fig5.add_trace(go.Scatter(y=monthly.values))
fig5.add_trace(go.Scatter(x=list(range(13,19)), y=forecast))

fig1.show()
fig2.show()
fig3.show()
fig4.show()
fig5.show()

customer.to_csv("segments.csv", index=False)
recommend.to_csv("recommendations.csv", index=False)
pivot.to_csv("pivot.csv")
corr.to_csv("corr.csv")

fig1.write_html("dash1.html")
fig2.write_html("dash2.html")

print(rmse_rf, rmse_xgb, acc)
print(recommend.head())
print(retention.head())

15688.129686789342 20340.28672364281 1.0
Empty DataFrame
Columns: [antecedents, consequents, antecedent support, consequent support, support, confidence, lift, representativity, leverage, conviction, zhangs_metric, jaccard, certainty, kulczynski]
Index: []
    Cohort  Index  Customer ID
0  2020-01      0           24
1  2020-01      1            1
2  2020-01      2            1
3  2020-01      4            2
4  2020-01      7            1
